In [136]:
import torch
import torch.nn as nn
import numpy as np
import torch.utils.data as Data

# 数据预处理
torch.util提供了许多实用的工具：
1. utils.data 数据加载
2. utils.tensorboard 可视化
3. utils.cpp_extension C++扩展

In [137]:
num_inputs = 2      #特征数量
num_examples = 1000 #样本个数
true_w = [2, -3.4]  #真实权重
true_b = 4.2        #真实偏差
features = torch.tensor(np.random.normal(0, 1, (num_examples, num_inputs)), dtype=torch.float32)
#创建了服从均值为0，标准差为1的正态分布，1000行，2列，转化为张量
labels = true_w[0] * features[:, 0] + true_w[1] * features[:, 1] + true_b
#生成标签，特征的第一列乘以第一个权重，加上特征的第二列乘以第二个权重，再加上偏差
labels += torch.tensor(np.random.normal(0, 0.01, size=labels.size()), dtype=torch.float32)
#给标签添加了一个小的随机噪声，使得生成的数据更像真实数据

In [138]:
batch_size = 10
dataset = Data.TensorDataset(features,labels)
data_iter = Data.DataLoader(dataset, batch_size, shuffle=True)
#随机读取10个数据样本的小批量



TensorDataset将多个张量包装成一个数据集，每个样本由每个张量中对应位置的元素组成。
结果：dataset 是一个可索引的对象

dataset[0] 返回 (features[0], labels[0]) - 第1个样本
dataset[1] 返回 (features[1], labels[1]) - 第2个样本
...
dataset[999] 返回 (features[999], labels[999]) - 第1000个样本


# 定义模型nn
nn定义了大量神将网络的层
nn利用autograd定义模型
nn的核心数据结构是Module，它既可以表示神经网络的某个层，也可以表示一个含有很多层的神经网络。
一个nn.Module实例应该包含一些层以及返回输出的前向传播方法

In [139]:
class LinearNet(nn.Module):
    def __init__(self, n_feature):
        super(LinearNet, self).__init__()
        self.linear = nn.Linear(n_feature, 1)
    def forward(self, x):
        y = self.linear(x)
        return y
net = LinearNet(num_inputs)
print(net)

LinearNet(
  (linear): Linear(in_features=2, out_features=1, bias=True)
)


In [140]:
for param in net.parameters():
    print(param)

Parameter containing:
tensor([[-0.0849, -0.1035]], requires_grad=True)
Parameter containing:
tensor([0.6098], requires_grad=True)


# 初始化模型参数
使用net之前，需要初始化模型参数，如线性模型的权重和偏置。
Pytorch在init模块中提供了多种参数初始化方法。
我们通过init.normal_将权重参数每个元素初始化为随机采样于均值为0，标准差为0.01的正态分布。

In [141]:
from torch.nn import init

init.normal_(net.linear.weight, mean=0, std=0.01)
init.constant_(net.linear.bias, val=0)  

Parameter containing:
tensor([0.], requires_grad=True)

# 定义损失函数

In [142]:
loss = nn.MSELoss()


# 定义优化算法
torch.optim模块提供了很多常用的优化算法如SGD，Adam和RMSProp等。
下面创建了一个用于优化net所有参数的优化器实例，并指定学习率为0.03的小批量随机梯度下降(SGD)为优化算法

In [143]:
import torch.optim as optim

optimizer = optim.SGD(net.parameters(), lr=0.03)
print(optimizer)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.03
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


**还可以为不同子网络设置不同的学习率，这在finetune（微调）时经常用到**

修改学习率的两种做法：
1. 修改optimizer.param_groups
2. 新建优化器(但是会丢失状态信息，发生意外情况)

In [144]:
# 调整学习率
for param_group in optimizer.param_groups:
    param_group['lr'] *= 0.1 # 学习率为之前的0.1倍

# 训练模型
通过调用optim实例的step函数迭代模型参数。
按照小批量随机梯度下降的定义，在step函数中指定批量大小，从而对批量中样本梯度求平均。

In [145]:
num_epochs = 50
for epoch in range(1, num_epochs + 1):
    for X, y in data_iter:
        output = net(X)
        l = loss(output, y.view(-1, 1))
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
    print('epoch %d, loss: %f' % (epoch, l.item()))

epoch 1, loss: 13.466887
epoch 2, loss: 1.794419
epoch 3, loss: 0.411867
epoch 4, loss: 0.273004
epoch 5, loss: 0.028867
epoch 6, loss: 0.033238
epoch 7, loss: 0.009074
epoch 8, loss: 0.001658
epoch 9, loss: 0.000659
epoch 10, loss: 0.000278
epoch 11, loss: 0.000077
epoch 12, loss: 0.000117
epoch 13, loss: 0.000198
epoch 14, loss: 0.000061
epoch 15, loss: 0.000103
epoch 16, loss: 0.000082
epoch 17, loss: 0.000084
epoch 18, loss: 0.000130
epoch 19, loss: 0.000087
epoch 20, loss: 0.000093
epoch 21, loss: 0.000064
epoch 22, loss: 0.000069
epoch 23, loss: 0.000113
epoch 24, loss: 0.000157
epoch 25, loss: 0.000055
epoch 26, loss: 0.000075
epoch 27, loss: 0.000184
epoch 28, loss: 0.000057
epoch 29, loss: 0.000130
epoch 30, loss: 0.000088
epoch 31, loss: 0.000084
epoch 32, loss: 0.000077
epoch 33, loss: 0.000078
epoch 34, loss: 0.000105
epoch 35, loss: 0.000111
epoch 36, loss: 0.000162
epoch 37, loss: 0.000103
epoch 38, loss: 0.000304
epoch 39, loss: 0.000219
epoch 40, loss: 0.000157
epoch 41

下面我们分别比较学到的模型参数和真实的模型参数。我们从net获得需要的层，并访问其权重（weight）和偏差（bias）。学到的参数和真实的参数很接近。

In [146]:
# 7. 查看训练结果
print("\n训练结果对比:")
print(f"真实权重: {true_w}")
print(f"训练权重: {net.linear.weight.data.squeeze().tolist()}")
print(f"真实偏置: {true_b}")
print(f"训练偏置: {net.linear.bias.data.item():.4f}")


训练结果对比:
真实权重: [2, -3.4]
训练权重: [1.9998561143875122, -3.400233268737793]
真实偏置: 4.2
训练偏置: 4.1996
